# Generalized Meta-Failure Predictor

This notebook extends the initial Meta-Failure Predictor
framework by evaluating cross-dataset and cross-architecture
generalization.


# Generalized Meta-Failure Predictor

## Objective

The objective of this project is to develop a generalized
meta-failure prediction framework capable of predicting
whether a machine learning model is likely to make an
incorrect prediction.

Unlike the previous version, which focused primarily on
a single dataset and architecture, this framework aims
to generalize across multiple datasets and multiple neural
network architectures.

The framework will be evaluated on an unseen
dataset-architecture combination to determine whether it
can learn generalized failure patterns rather than
dataset-specific or architecture-specific characteristics.

---

## Datasets

1. CIFAR-10
2. Fashion-MNIST

---

## Architectures

1. ResNet18
2. MobileNetV2

---

## Reliability Signals

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

---

## Meta-Model

Random Forest Classifier

---

## Unseen Evaluation

The combination:

Fashion-MNIST + MobileNetV2

will be excluded from meta-model training and used only
for final evaluation.

This experiment will assess cross-dataset and
cross-architecture generalization.

In [1]:
import torch
import torchvision
import sklearn
import pandas as pd
import numpy as np

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("MPS Available:", torch.backends.mps.is_available())

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

PyTorch: 2.12.0
Torchvision: 0.27.0
MPS Available: True
Device: mps


# Dataset Preparation

## Objective

The objective of this phase is to load multiple datasets
that will be used to evaluate cross-dataset
generalization.

Two datasets are selected:

1. CIFAR-10
2. Fashion-MNIST

These datasets represent different image domains and
will help assess whether the proposed framework can
learn generalized failure patterns.

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# CIFAR-10 Transform
cifar_transform = transforms.Compose([
    transforms.ToTensor()
])

# Fashion-MNIST Transform
fashion_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

# CIFAR-10
cifar_train = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=cifar_transform
)

cifar_test = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=cifar_transform
)

# Fashion-MNIST
fashion_train = datasets.FashionMNIST(
    root="../data",
    train=True,
    download=True,
    transform=fashion_transform
)

fashion_test = datasets.FashionMNIST(
    root="../data",
    train=False,
    download=True,
    transform=fashion_transform
)

print("CIFAR Train:", len(cifar_train))
print("CIFAR Test:", len(cifar_test))

print("Fashion Train:", len(fashion_train))
print("Fashion Test:", len(fashion_test))

100%|██████████| 26.4M/26.4M [00:04<00:00, 5.36MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 205kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.98MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.37MB/s]

CIFAR Train: 50000
CIFAR Test: 10000
Fashion Train: 60000
Fashion Test: 10000


# DataLoader Creation

## Objective

Create efficient data pipelines for training and testing.

DataLoaders enable batch-wise loading of images and labels,
allowing the models to train efficiently on the available
hardware.

In [3]:
from torch.utils.data import DataLoader

BATCH_SIZE = 128

# CIFAR-10
cifar_train_loader = DataLoader(
    cifar_train,
    batch_size=BATCH_SIZE,
    shuffle=True
)

cifar_test_loader = DataLoader(
    cifar_test,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Fashion-MNIST
fashion_train_loader = DataLoader(
    fashion_train,
    batch_size=BATCH_SIZE,
    shuffle=True
)

fashion_test_loader = DataLoader(
    fashion_test,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("CIFAR Train Batches:", len(cifar_train_loader))
print("CIFAR Test Batches:", len(cifar_test_loader))

print("Fashion Train Batches:", len(fashion_train_loader))
print("Fashion Test Batches:", len(fashion_test_loader))

CIFAR Train Batches: 391
CIFAR Test Batches: 79
Fashion Train Batches: 469
Fashion Test Batches: 79


# Base Model Training Strategy

## Objective

To evaluate cross-dataset and cross-architecture
generalization, the meta-model will be trained using
failure data collected from multiple datasets and multiple
architectures.

Training Combinations:

1. CIFAR-10 + ResNet18
2. CIFAR-10 + MobileNetV2
3. Fashion-MNIST + ResNet18

Unseen Evaluation Combination:

1. Fashion-MNIST + MobileNetV2

The unseen combination will be excluded from meta-model
training and used only for final evaluation.

# Base Model Preparation

## Objective

Reuse previously trained models where possible to reduce
computational cost while maintaining experimental validity.

The previously trained ResNet18 model on CIFAR-10 will be
reused as one of the training combinations.

Remaining model-dataset combinations will be trained and
evaluated separately.

In [4]:
import torch
import torchvision.models as models

# Create model
resnet18_cifar = models.resnet18(weights=None)

resnet18_cifar.fc = torch.nn.Linear(
    resnet18_cifar.fc.in_features,
    10
)

# Load saved weights
resnet18_cifar.load_state_dict(
    torch.load(
        "../models/resnet18_cifar10.pth",
        map_location=device
    )
)

resnet18_cifar = resnet18_cifar.to(device)

print("ResNet18 CIFAR10 Loaded Successfully")

ResNet18 CIFAR10 Loaded Successfully


# MobileNetV2 on CIFAR-10

## Objective

Train a second neural network architecture on CIFAR-10 to
introduce architectural diversity into the meta-failure
prediction framework.

Unlike ResNet18, MobileNetV2 uses depthwise separable
convolutions and a lightweight architecture design.

Training multiple architectures enables evaluation of
cross-architecture failure prediction generalization.

In [5]:
import torchvision.models as models
import torch.nn as nn

mobilenet_cifar = models.mobilenet_v2(weights=None)

mobilenet_cifar.classifier[1] = nn.Linear(
    mobilenet_cifar.classifier[1].in_features,
    10
)

mobilenet_cifar = mobilenet_cifar.to(device)

print(mobilenet_cifar)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

## MobileNetV2 Training

### Objective

Train MobileNetV2 on CIFAR-10 and evaluate its classification
performance.

This model introduces architectural diversity into the
framework and will later contribute failure samples for
training the generalized meta-failure predictor.

### Training Configuration

- Optimizer: Adam
- Loss Function: Cross Entropy Loss
- Epochs: 5
- Device: Apple GPU (MPS)

### Expected Outcome

The trained MobileNetV2 model should achieve reasonable
classification accuracy on CIFAR-10 while providing a
different failure distribution compared to ResNet18.

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mobilenet_cifar.parameters(),
    lr=0.001
)

EPOCHS = 5

for epoch in range(EPOCHS):

    mobilenet_cifar.train()

    running_loss = 0

    loop = tqdm(
        cifar_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilenet_cifar(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    mobilenet_cifar.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in cifar_test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = mobilenet_cifar(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(cifar_train_loader):.4f} "
        f"Accuracy={accuracy:.2f}%"
    )

Epoch 1/5: 100%|██████████| 391/391 [00:15<00:00, 25.82it/s]


Epoch 1: Loss=1.9188 Accuracy=39.03%


Epoch 2/5: 100%|██████████| 391/391 [00:11<00:00, 34.65it/s]


Epoch 2: Loss=1.5544 Accuracy=44.42%


Epoch 3/5: 100%|██████████| 391/391 [00:11<00:00, 34.57it/s]


Epoch 3: Loss=1.4073 Accuracy=49.14%


Epoch 4/5: 100%|██████████| 391/391 [00:11<00:00, 34.40it/s]


Epoch 4: Loss=1.2910 Accuracy=54.56%


Epoch 5/5: 100%|██████████| 391/391 [00:11<00:00, 34.09it/s]


Epoch 5: Loss=1.1953 Accuracy=57.01%


## MobileNetV2 CIFAR-10 Results

### Classification Performance

| Epoch | Accuracy |
|---------|---------|
| 1 | 39.03% |
| 2 | 44.42% |
| 3 | 49.14% |
| 4 | 54.56% |
| 5 | 57.01% |

### Observation

MobileNetV2 successfully learned CIFAR-10 classification
patterns and achieved 57.01% test accuracy after five
epochs.

Although the performance is lower than ResNet18, the model
introduces architectural diversity into the framework and
provides a different distribution of prediction failures.

These diverse failure patterns are valuable for training
a generalized meta-failure predictor.

In [7]:
torch.save(
    mobilenet_cifar.state_dict(),
    "../models/mobilenetv2_cifar10.pth"
)

print("MobileNetV2 CIFAR10 Saved!")

MobileNetV2 CIFAR10 Saved!


# Generalized Reliability Feature Extraction

## Objective

Extract architecture-independent reliability signals that
can be computed from any neural network architecture.

Unlike the previous framework, which relied on raw
activation vectors and PCA compression, this version uses
statistical summaries of internal activations.

Extracted Features:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

These features enable the development of a generalized
meta-failure prediction framework across multiple
architectures.

In [8]:
feature_extractor_resnet = torch.nn.Sequential(
    *list(resnet18_cifar.children())[:-1]
)

print(feature_extractor_resnet)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (

## ResNet18 + CIFAR10 Feature Extraction

### Objective

Generate architecture-independent reliability features
from the trained ResNet18 model.

The extracted features will be used to train the
generalized meta-failure predictor.

For each test sample, statistical summaries of the
internal activations are computed instead of storing
the complete activation vector.

This approach enables compatibility across different
neural network architectures.

In [15]:
import torch.nn.functional as F
import pandas as pd
import numpy as np

resnet18_cifar.eval()

confidences = []
entropies = []

activation_means = []
activation_stds = []
activation_maxs = []
activation_mins = []

failures = []

with torch.no_grad():

    for images, labels in cifar_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet18_cifar(images)

        probs = F.softmax(
            outputs,
            dim=1
        )

        confidence, predictions = torch.max(
            probs,
            dim=1
        )

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-10),
            dim=1
        )

        activations = feature_extractor_resnet(
            images
        )

        activations = activations.view(
            activations.size(0),
            -1
        )

        activation_means.extend(
            activations.mean(dim=1)
            .cpu()
            .numpy()
        )

        activation_stds.extend(
            activations.std(dim=1)
            .cpu()
            .numpy()
        )

        activation_maxs.extend(
            activations.max(dim=1)[0]
            .cpu()
            .numpy()
        )

        activation_mins.extend(
            activations.min(dim=1)[0]
            .cpu()
            .numpy()
        )

        confidences.extend(
            confidence.cpu().numpy()
        )

        entropies.extend(
            entropy.cpu().numpy()
        )

        failures.extend(
            (predictions != labels)
            .int()
            .cpu()
            .numpy()
        )

resnet_cifar_df = pd.DataFrame({
    "confidence": confidences,
    "entropy": entropies,
    "activation_mean": activation_means,
    "activation_std": activation_stds,
    "activation_max": activation_maxs,
    "activation_min": activation_mins,
    "failure": failures
})

print(resnet_cifar_df.head())
print()
print("Shape:", resnet_cifar_df.shape)
print("Failures:", resnet_cifar_df["failure"].sum())

   confidence   entropy  activation_mean  activation_std  activation_max  \
0    0.693736  1.112976         0.370606        0.489552        3.145806   
1    0.684337  0.868562         1.074382        1.307902        6.743712   
2    0.825382  0.580671         0.695826        0.833168        4.562689   
3    0.971878  0.170356         0.510351        0.751364        3.695982   
4    0.759581  0.774034         0.608280        0.922197        4.602489   

   activation_min  failure  
0             0.0        0  
1             0.0        0  
2             0.0        0  
3             0.0        0  
4             0.0        0  

Shape: (10000, 7)
Failures: 2869


In [11]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Correct CIFAR-10 transform (same as Version 1)
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

# Reload CIFAR-10
cifar_test = datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=cifar_transform
)

cifar_test_loader = DataLoader(
    cifar_test,
    batch_size=128,
    shuffle=False
)

print("CIFAR Test Reloaded")

CIFAR Test Reloaded


In [12]:
correct = 0
total = 0

resnet18_cifar.eval()

with torch.no_grad():

    for images, labels in cifar_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet18_cifar(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 71.31


In [13]:
print(feature_extractor_resnet)

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (

## ResNet18 + CIFAR10 Feature Extraction Results

### Generated Features

For each test sample, the following reliability signals were extracted:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

### Dataset Statistics

- Total Samples: 10,000
- Correct Predictions: 7,131
- Incorrect Predictions: 2,869

### Observation

The extracted feature dataset successfully captures both
prediction-level uncertainty information and internal
network behavior.

These architecture-independent features will be used for
training the generalized meta-failure predictor.

In [16]:
resnet_cifar_df.to_csv(
    "../features/resnet18_cifar10_features.csv",
    index=False
)

print("Saved!")

Saved!


In [17]:
print(mobilenet_cifar)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
  

## MobileNetV2 Feature Extraction Preparation

### Objective

Create a feature extraction pipeline for MobileNetV2.

The extracted activations will be converted into
architecture-independent statistical features:

- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

These features will be combined with confidence and entropy
to generate a generalized failure dataset.

In [18]:
mobilenet_feature_extractor = mobilenet_cifar.features

print("Feature Extractor Ready")

Feature Extractor Ready


In [19]:
images, labels = next(iter(cifar_test_loader))

images = images.to(device)

with torch.no_grad():

    activations = mobilenet_feature_extractor(images)

print("Activation Shape:", activations.shape)

Activation Shape: torch.Size([128, 1280, 1, 1])


## MobileNetV2 Activation Analysis

### Observation

The extracted MobileNetV2 activation tensor has shape:

(128, 1280, 1, 1)

This differs from the ResNet18 activation representation,
which contains 512 channels.

This observation highlights a key limitation of using raw
activation vectors for failure prediction because feature
dimensionality varies across architectures.

To address this issue, activation tensors are summarized
using statistical descriptors:

- Mean
- Standard Deviation
- Maximum
- Minimum

These architecture-independent features enable the
development of a generalized meta-failure prediction
framework.

In [20]:
import torch.nn.functional as F
import pandas as pd
import numpy as np

mobilenet_cifar.eval()

confidences = []
entropies = []

activation_means = []
activation_stds = []
activation_maxs = []
activation_mins = []

failures = []

with torch.no_grad():

    for images, labels in cifar_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilenet_cifar(images)

        probs = F.softmax(
            outputs,
            dim=1
        )

        confidence, predictions = torch.max(
            probs,
            dim=1
        )

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-10),
            dim=1
        )

        activations = mobilenet_feature_extractor(
            images
        )

        activations = activations.view(
            activations.size(0),
            -1
        )

        activation_means.extend(
            activations.mean(dim=1)
            .cpu()
            .numpy()
        )

        activation_stds.extend(
            activations.std(dim=1)
            .cpu()
            .numpy()
        )

        activation_maxs.extend(
            activations.max(dim=1)[0]
            .cpu()
            .numpy()
        )

        activation_mins.extend(
            activations.min(dim=1)[0]
            .cpu()
            .numpy()
        )

        confidences.extend(
            confidence.cpu().numpy()
        )

        entropies.extend(
            entropy.cpu().numpy()
        )

        failures.extend(
            (predictions != labels)
            .int()
            .cpu()
            .numpy()
        )

mobilenet_cifar_df = pd.DataFrame({
    "confidence": confidences,
    "entropy": entropies,
    "activation_mean": activation_means,
    "activation_std": activation_stds,
    "activation_max": activation_maxs,
    "activation_min": activation_mins,
    "failure": failures
})

print(mobilenet_cifar_df.head())
print()
print("Shape:", mobilenet_cifar_df.shape)
print("Failures:", mobilenet_cifar_df["failure"].sum())

   confidence   entropy  activation_mean  activation_std  activation_max  \
0    0.264273  1.905313         0.174318        0.325949        1.859067   
1    0.667097  1.053511         0.201999        0.287282        1.591721   
2    0.686431  0.869989         0.277519        0.490864        1.643954   
3    0.564086  1.297502         0.153230        0.237153        1.489510   
4    0.898178  0.448239         0.248521        0.346367        2.256601   

   activation_min  failure  
0             0.0        1  
1             0.0        0  
2             0.0        1  
3             0.0        1  
4             0.0        0  

Shape: (10000, 7)
Failures: 6967


In [21]:
correct = 0
total = 0

mobilenet_cifar.eval()

with torch.no_grad():

    for images, labels in cifar_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilenet_cifar(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

Accuracy: 30.33


## MobileNetV2 Retraining Notice

### Observation

An inconsistency was identified between the preprocessing
pipeline used during MobileNetV2 training and the pipeline
used during evaluation.

The initial MobileNetV2 model was trained before CIFAR-10
normalization was introduced into the workflow.

As a result, evaluation performance dropped significantly
when tested using normalized inputs.

### Corrective Action

To ensure experimental consistency, MobileNetV2 will be
retrained using the same normalized CIFAR-10 preprocessing
pipeline employed by ResNet18.

This guarantees that all architectures are trained and
evaluated under identical conditions.

In [22]:
import torchvision.models as models
import torch.nn as nn

mobilenet_cifar = models.mobilenet_v2(weights=None)

mobilenet_cifar.classifier[1] = nn.Linear(
    mobilenet_cifar.classifier[1].in_features,
    10
)

mobilenet_cifar = mobilenet_cifar.to(device)

print("Fresh MobileNetV2 Created")

Fresh MobileNetV2 Created


In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mobilenet_cifar.parameters(),
    lr=0.001
)

EPOCHS = 5

for epoch in range(EPOCHS):

    mobilenet_cifar.train()

    running_loss = 0

    loop = tqdm(
        cifar_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilenet_cifar(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    mobilenet_cifar.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in cifar_test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = mobilenet_cifar(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(cifar_train_loader):.4f} "
        f"Accuracy={accuracy:.2f}%"
    )

Epoch 1/5: 100%|██████████| 391/391 [00:11<00:00, 34.70it/s]


Epoch 1: Loss=1.9555 Accuracy=16.01%


Epoch 2/5: 100%|██████████| 391/391 [00:11<00:00, 34.70it/s]


Epoch 2: Loss=1.5849 Accuracy=18.97%


Epoch 3/5: 100%|██████████| 391/391 [00:11<00:00, 34.41it/s]


Epoch 3: Loss=1.4214 Accuracy=24.67%


Epoch 4/5: 100%|██████████| 391/391 [00:11<00:00, 33.84it/s]


Epoch 4: Loss=1.3010 Accuracy=24.28%


Epoch 5/5: 100%|██████████| 391/391 [00:11<00:00, 33.37it/s]


Epoch 5: Loss=1.2016 Accuracy=20.38%


In [24]:
print(cifar_train.transform)

Compose(
    ToTensor()
)


In [25]:
print(cifar_test.transform)

Compose(
    ToTensor()
    Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.201))
)


In [26]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Correct CIFAR transform
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2023, 0.1994, 0.2010)
    )
])

# Recreate BOTH train and test datasets
cifar_train = datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=cifar_transform
)

cifar_test = datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=cifar_transform
)

# Recreate BOTH dataloaders
cifar_train_loader = DataLoader(
    cifar_train,
    batch_size=128,
    shuffle=True
)

cifar_test_loader = DataLoader(
    cifar_test,
    batch_size=128,
    shuffle=False
)

print("CIFAR loaders recreated successfully")

CIFAR loaders recreated successfully


In [27]:
print(cifar_train.transform)
print(cifar_test.transform)

Compose(
    ToTensor()
    Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.201))
)
Compose(
    ToTensor()
    Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.2023, 0.1994, 0.201))
)


In [28]:
import torchvision.models as models
import torch.nn as nn

mobilenet_cifar = models.mobilenet_v2(weights=None)

mobilenet_cifar.classifier[1] = nn.Linear(
    mobilenet_cifar.classifier[1].in_features,
    10
)

mobilenet_cifar = mobilenet_cifar.to(device)

print("Fresh MobileNetV2 Created")

Fresh MobileNetV2 Created


In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mobilenet_cifar.parameters(),
    lr=0.001
)

EPOCHS = 5

for epoch in range(EPOCHS):

    mobilenet_cifar.train()

    running_loss = 0

    loop = tqdm(
        cifar_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilenet_cifar(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    mobilenet_cifar.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in cifar_test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = mobilenet_cifar(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(cifar_train_loader):.4f} "
        f"Accuracy={accuracy:.2f}%"
    )

Epoch 1/5: 100%|██████████| 391/391 [00:11<00:00, 32.94it/s]


Epoch 1: Loss=1.9472 Accuracy=38.10%


Epoch 2/5: 100%|██████████| 391/391 [00:12<00:00, 32.42it/s]


Epoch 2: Loss=1.5520 Accuracy=46.41%


Epoch 3/5: 100%|██████████| 391/391 [00:12<00:00, 31.11it/s]


Epoch 3: Loss=1.3954 Accuracy=50.74%


Epoch 4/5: 100%|██████████| 391/391 [00:13<00:00, 28.60it/s]


Epoch 4: Loss=1.2787 Accuracy=54.39%


Epoch 5/5: 100%|██████████| 391/391 [00:16<00:00, 23.88it/s]


Epoch 5: Loss=1.1789 Accuracy=57.20%


## MobileNetV2 CIFAR-10 Results

### Classification Performance

Final Test Accuracy: 57.20%

### Observation

After correcting the preprocessing pipeline, MobileNetV2
achieved stable performance on CIFAR-10.

The model provides an alternative architectural perspective
for failure prediction and contributes additional failure
patterns for training the generalized meta-failure
predictor.

Combined with ResNet18, the framework now contains failure
information from two distinct neural network architectures.

In [30]:
torch.save(
    mobilenet_cifar.state_dict(),
    "../models/mobilenetv2_cifar10.pth"
)

print("Correct MobileNetV2 Saved!")

Correct MobileNetV2 Saved!


In [31]:
mobilenet_feature_extractor = mobilenet_cifar.features

print("Feature Extractor Ready")

Feature Extractor Ready


In [32]:
import torch.nn.functional as F
import pandas as pd
import numpy as np

mobilenet_cifar.eval()

confidences = []
entropies = []

activation_means = []
activation_stds = []
activation_maxs = []
activation_mins = []

failures = []

with torch.no_grad():

    for images, labels in cifar_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilenet_cifar(images)

        probs = F.softmax(
            outputs,
            dim=1
        )

        confidence, predictions = torch.max(
            probs,
            dim=1
        )

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-10),
            dim=1
        )

        activations = mobilenet_feature_extractor(
            images
        )

        activations = activations.view(
            activations.size(0),
            -1
        )

        activation_means.extend(
            activations.mean(dim=1)
            .cpu()
            .numpy()
        )

        activation_stds.extend(
            activations.std(dim=1)
            .cpu()
            .numpy()
        )

        activation_maxs.extend(
            activations.max(dim=1)[0]
            .cpu()
            .numpy()
        )

        activation_mins.extend(
            activations.min(dim=1)[0]
            .cpu()
            .numpy()
        )

        confidences.extend(
            confidence.cpu().numpy()
        )

        entropies.extend(
            entropy.cpu().numpy()
        )

        failures.extend(
            (predictions != labels)
            .int()
            .cpu()
            .numpy()
        )

mobilenet_cifar_df = pd.DataFrame({
    "confidence": confidences,
    "entropy": entropies,
    "activation_mean": activation_means,
    "activation_std": activation_stds,
    "activation_max": activation_maxs,
    "activation_min": activation_mins,
    "failure": failures
})

print(mobilenet_cifar_df.head())
print()
print("Shape:", mobilenet_cifar_df.shape)
print("Failures:", mobilenet_cifar_df["failure"].sum())

   confidence   entropy  activation_mean  activation_std  activation_max  \
0    0.576962  1.231839         0.130839        0.240987        1.660688   
1    0.714289  0.899934         0.228657        0.392864        2.332921   
2    0.365298  1.485339         0.144793        0.232916        1.222224   
3    0.681728  1.024587         0.170095        0.247431        1.573043   
4    0.745226  0.820578         0.247805        0.345342        2.069619   

   activation_min  failure  
0             0.0        0  
1             0.0        0  
2             0.0        0  
3             0.0        1  
4             0.0        0  

Shape: (10000, 7)
Failures: 4280


## MobileNetV2 + CIFAR10 Feature Extraction Results

### Generated Features

For each test sample, the following reliability signals were extracted:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

### Dataset Statistics

- Total Samples: 10,000
- Correct Predictions: 5,720
- Incorrect Predictions: 4,280

### Observation

The generated feature dataset captures both uncertainty
information and internal activation behavior from
MobileNetV2.

Combined with the ResNet18 feature dataset, the framework
now contains failure information from two distinct neural
network architectures trained on the same dataset.

In [33]:
mobilenet_cifar_df.to_csv(
    "../features/mobilenetv2_cifar10_features.csv",
    index=False
)

print("Saved!")

Saved!


# Cross-Dataset Generalization

## Objective

Evaluate whether the proposed reliability features remain
useful when the image domain changes.

Fashion-MNIST differs significantly from CIFAR-10 because
it contains grayscale clothing images rather than natural
colored object images.

Training models on multiple datasets allows the framework
to learn failure patterns that are less dependent on a
specific dataset and more representative of general model
behavior.

In [34]:
images, labels = next(iter(fashion_train_loader))

print("Image Shape:", images.shape)
print("Label Shape:", labels.shape)

Image Shape: torch.Size([128, 3, 32, 32])
Label Shape: torch.Size([128])


## Fashion-MNIST Verification

### Observation

Fashion-MNIST images were successfully transformed into a
format compatible with the selected deep learning models.

Original Format:

- 28 × 28 × 1 (Grayscale)

Transformed Format:

- 32 × 32 × 3 (RGB)

This preprocessing step ensures that the same neural
network architectures can be evaluated across multiple
datasets without architectural modifications.

In [36]:
import torchvision.models as models
import torch.nn as nn

resnet18_fashion = models.resnet18(weights=None)

resnet18_fashion.fc = nn.Linear(
    resnet18_fashion.fc.in_features,
    10
)

resnet18_fashion = resnet18_fashion.to(device)

print(resnet18_fashion)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

# ResNet18 on Fashion-MNIST

## Objective

Train ResNet18 on Fashion-MNIST to evaluate how failure
patterns change when the dataset domain changes while
keeping the architecture constant.

This experiment enables analysis of dataset
generalization by comparing:

- ResNet18 + CIFAR10
- ResNet18 + FashionMNIST

The resulting failure data will contribute to the
generalized meta-failure predictor training set.

In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet18_fashion.parameters(),
    lr=0.001
)

EPOCHS = 5

for epoch in range(EPOCHS):

    resnet18_fashion.train()

    running_loss = 0

    loop = tqdm(
        fashion_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18_fashion(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    resnet18_fashion.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in fashion_test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18_fashion(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(fashion_train_loader):.4f} "
        f"Accuracy={accuracy:.2f}%"
    )

Epoch 1/5: 100%|██████████| 469/469 [00:18<00:00, 24.69it/s]


Epoch 1: Loss=0.4151 Accuracy=87.00%


Epoch 2/5: 100%|██████████| 469/469 [00:17<00:00, 26.25it/s]


Epoch 2: Loss=0.2956 Accuracy=88.64%


Epoch 3/5: 100%|██████████| 469/469 [00:19<00:00, 23.74it/s]


Epoch 3: Loss=0.2574 Accuracy=89.06%


Epoch 4/5: 100%|██████████| 469/469 [00:23<00:00, 19.82it/s]


Epoch 4: Loss=0.2365 Accuracy=89.23%


Epoch 5/5: 100%|██████████| 469/469 [00:29<00:00, 16.09it/s]


Epoch 5: Loss=0.2139 Accuracy=89.92%


## ResNet18 + Fashion-MNIST Results

### Classification Performance

| Epoch | Accuracy |
|--------|--------|
| 1 | 87.00% |
| 2 | 88.64% |
| 3 | 89.06% |
| 4 | 89.23% |
| 5 | 89.92% |

### Observation

ResNet18 achieved strong performance on Fashion-MNIST,
reaching 89.92% test accuracy after five epochs.

Compared to CIFAR-10, Fashion-MNIST is a simpler image
classification task, resulting in a lower failure rate.

The resulting failure dataset introduces cross-dataset
diversity into the generalized meta-failure prediction
framework.

In [38]:
torch.save(
    resnet18_fashion.state_dict(),
    "../models/resnet18_fashionmnist.pth"
)

print("ResNet18 FashionMNIST Saved!")

ResNet18 FashionMNIST Saved!


In [39]:
feature_extractor_fashion = torch.nn.Sequential(
    *list(resnet18_fashion.children())[:-1]
)

print("Feature Extractor Ready")

Feature Extractor Ready


## ResNet18 + Fashion-MNIST Feature Extraction

### Objective

Extract architecture-independent reliability signals from
the Fashion-MNIST model.

These features will later be combined with the CIFAR-10
feature datasets to construct the generalized
meta-failure prediction dataset.

Extracted Features:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

In [40]:
import torch.nn.functional as F
import pandas as pd
import numpy as np

resnet18_fashion.eval()

confidences = []
entropies = []

activation_means = []
activation_stds = []
activation_maxs = []
activation_mins = []

failures = []

with torch.no_grad():

    for images, labels in fashion_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = resnet18_fashion(images)

        probs = F.softmax(outputs, dim=1)

        confidence, predictions = torch.max(
            probs,
            dim=1
        )

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-10),
            dim=1
        )

        activations = feature_extractor_fashion(images)

        activations = activations.view(
            activations.size(0),
            -1
        )

        activation_means.extend(
            activations.mean(dim=1)
            .cpu()
            .numpy()
        )

        activation_stds.extend(
            activations.std(dim=1)
            .cpu()
            .numpy()
        )

        activation_maxs.extend(
            activations.max(dim=1)[0]
            .cpu()
            .numpy()
        )

        activation_mins.extend(
            activations.min(dim=1)[0]
            .cpu()
            .numpy()
        )

        confidences.extend(
            confidence.cpu().numpy()
        )

        entropies.extend(
            entropy.cpu().numpy()
        )

        failures.extend(
            (predictions != labels)
            .int()
            .cpu()
            .numpy()
        )

resnet_fashion_df = pd.DataFrame({
    "confidence": confidences,
    "entropy": entropies,
    "activation_mean": activation_means,
    "activation_std": activation_stds,
    "activation_max": activation_maxs,
    "activation_min": activation_mins,
    "failure": failures
})

print(resnet_fashion_df.head())
print()
print("Shape:", resnet_fashion_df.shape)
print("Failures:", resnet_fashion_df["failure"].sum())

   confidence   entropy  activation_mean  activation_std  activation_max  \
0    0.998366  0.012513         1.205912        1.419986        5.997708   
1    0.999800  0.002109         0.744657        1.406017        7.946265   
2    0.999990  0.000146         1.892720        1.786598        7.090538   
3    0.999997  0.000048         2.076962        1.927658        7.562252   
4    0.670650  0.909583         0.295683        0.712061        4.482496   

   activation_min  failure  
0             0.0        0  
1             0.0        0  
2             0.0        0  
3             0.0        0  
4             0.0        1  

Shape: (10000, 7)
Failures: 1008


## ResNet18 + Fashion-MNIST Feature Extraction Results

### Generated Features

For each test sample, the following reliability signals were extracted:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

### Dataset Statistics

- Total Samples: 10,000
- Correct Predictions: 8,992
- Incorrect Predictions: 1,008

### Observation

The Fashion-MNIST feature dataset exhibits a significantly
lower failure rate compared to CIFAR-10.

This difference introduces valuable diversity into the
meta-dataset and enables the meta-model to learn failure
patterns across multiple image domains rather than relying
on dataset-specific characteristics.

In [41]:
resnet_fashion_df.to_csv(
    "../features/resnet18_fashionmnist_features.csv",
    index=False
)

print("Saved!")

Saved!


# Generalized Meta-Dataset Construction

## Objective

Combine failure datasets generated from multiple datasets
and multiple architectures into a single unified
meta-dataset.

Training Sources:

1. ResNet18 + CIFAR10
2. MobileNetV2 + CIFAR10
3. ResNet18 + FashionMNIST

The resulting dataset will be used to train a generalized
meta-failure predictor capable of identifying potential
prediction failures across diverse model and dataset
combinations.

Input Features:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

Target Variable:

- Failure

In [43]:
import pandas as pd

meta_df = pd.concat(
    [
        resnet_cifar_df,
        mobilenet_cifar_df,
        resnet_fashion_df
    ],
    ignore_index=True
)

print("Meta Dataset Shape:", meta_df.shape)
print()

print(meta_df["failure"].value_counts())

Meta Dataset Shape: (30000, 7)

failure
0    21843
1     8157
Name: count, dtype: int64


## Unified Meta-Dataset Statistics

### Total Samples

30,000

### Feature Count

6 reliability features

### Target Variable

Failure

### Observation

The unified meta-dataset combines information from
multiple datasets and architectures, enabling the
meta-model to learn generalized failure patterns rather
than relying on architecture-specific or dataset-specific
characteristics.

In [44]:
from sklearn.model_selection import train_test_split

X = meta_df.drop(
    columns=["failure"]
)

y = meta_df["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (24000, 6)
Test Shape: (6000, 6)


In [45]:
from sklearn.ensemble import RandomForestClassifier

meta_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

meta_model.fit(
    X_train,
    y_train
)

print("Generalized Meta Model Trained!")

Generalized Meta Model Trained!


## Meta-Dataset Construction Results

### Dataset Size

- Total Samples: 30,000
- Training Samples: 24,000
- Testing Samples: 6,000

### Failure Distribution

- Correct Predictions (Failure = 0): 21,843
- Incorrect Predictions (Failure = 1): 8,157

### Observation

The combined dataset contains reliability information
from multiple datasets and multiple neural network
architectures.

This diversity enables the meta-model to learn
generalized failure patterns instead of memorizing
architecture-specific behavior.

In [46]:
from sklearn.ensemble import RandomForestClassifier

meta_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

meta_model.fit(
    X_train,
    y_train
)

print("Generalized Meta Model Trained!")

Generalized Meta Model Trained!


In [47]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

preds = meta_model.predict(X_test)

print(
    "Meta Accuracy:",
    accuracy_score(
        y_test,
        preds
    )
)

print(
    classification_report(
        y_test,
        preds
    )
)

Meta Accuracy: 0.7705
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      4369
           1       0.59      0.51      0.55      1631

    accuracy                           0.77      6000
   macro avg       0.71      0.69      0.70      6000
weighted avg       0.76      0.77      0.76      6000



## Generalized Meta-Model Results

### Random Forest Performance

| Metric | Value |
|----------|----------|
| Accuracy | 77.05% |
| Failure Precision | 59% |
| Failure Recall | 51% |
| Failure F1-Score | 55% |

### Observation

The generalized meta-model successfully learned failure
patterns from multiple datasets and architectures.

The model achieved 77.05% overall accuracy while correctly
identifying more than half of the actual failures.

This result demonstrates that architecture-independent
reliability features contain meaningful information for
predicting model failures across diverse settings.

### Interpretation

The meta-model is able to distinguish between likely
correct and likely incorrect predictions using only:

- Confidence
- Entropy
- Activation Mean
- Activation Standard Deviation
- Activation Maximum
- Activation Minimum

without requiring architecture-specific activation vectors.

# Explainable AI Analysis

## Objective

Analyze the importance of each reliability feature in the
generalized meta-failure predictor.

Feature importance analysis helps identify which signals
contribute most to failure prediction and improves the
interpretability of the framework.

In [48]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": meta_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

print(feature_importance)

           feature  importance
0       confidence    0.273936
1          entropy    0.231759
3   activation_std    0.173670
4   activation_max    0.166690
2  activation_mean    0.153946
5   activation_min    0.000000


## Feature Importance Analysis

### Reliability Signal Ranking

| Feature | Importance |
|----------|----------|
| Confidence | 0.274 |
| Entropy | 0.232 |
| Activation Standard Deviation | 0.174 |
| Activation Maximum | 0.167 |
| Activation Mean | 0.154 |
| Activation Minimum | 0.000 |

### Observation

Confidence and entropy are the most influential features
for predicting failures.

Together, they contribute more than 50% of the total
decision-making power of the meta-model.

Activation-based statistics also contribute meaningful
information, indicating that internal neural network
behavior contains useful failure-related signals beyond
prediction confidence alone.

Activation minimum contributes almost no predictive value,
suggesting that this feature may be removed in future
versions of the framework.

# Unseen Dataset-Architecture Evaluation

## Objective

Evaluate the generalized meta-failure predictor on a
dataset-architecture combination that was completely
excluded from meta-model training.

Training Combinations:

- ResNet18 + CIFAR10
- MobileNetV2 + CIFAR10
- ResNet18 + FashionMNIST

Unseen Evaluation Combination:

- MobileNetV2 + FashionMNIST

This experiment assesses whether the proposed framework
can generalize beyond the combinations observed during
training.

In [50]:
import torchvision.models as models
import torch.nn as nn

mobilenet_fashion = models.mobilenet_v2(weights=None)

mobilenet_fashion.classifier[1] = nn.Linear(
    mobilenet_fashion.classifier[1].in_features,
    10
)

mobilenet_fashion = mobilenet_fashion.to(device)

print("MobileNetV2 FashionMNIST Created")

MobileNetV2 FashionMNIST Created


## MobileNetV2 on Fashion-MNIST

### Objective

Train MobileNetV2 on Fashion-MNIST to create a completely
unseen dataset-architecture combination.

The resulting failure data will NOT be used during
meta-model training and will instead serve as the final
generalization evaluation benchmark.

In [51]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    mobilenet_fashion.parameters(),
    lr=0.001
)

EPOCHS = 5

for epoch in range(EPOCHS):

    mobilenet_fashion.train()

    running_loss = 0

    loop = tqdm(
        fashion_train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = mobilenet_fashion(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    mobilenet_fashion.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in fashion_test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = mobilenet_fashion(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch+1}: "
        f"Loss={running_loss/len(fashion_train_loader):.4f} "
        f"Accuracy={accuracy:.2f}%"
    )

Epoch 1/5: 100%|██████████| 469/469 [00:14<00:00, 31.56it/s]


Epoch 1: Loss=0.7121 Accuracy=81.44%


Epoch 2/5: 100%|██████████| 469/469 [00:14<00:00, 32.17it/s]


Epoch 2: Loss=0.4070 Accuracy=85.35%


Epoch 3/5: 100%|██████████| 469/469 [00:14<00:00, 31.78it/s]


Epoch 3: Loss=0.3479 Accuracy=86.13%


Epoch 4/5: 100%|██████████| 469/469 [00:15<00:00, 31.26it/s]


Epoch 4: Loss=0.3193 Accuracy=87.43%


Epoch 5/5: 100%|██████████| 469/469 [00:15<00:00, 30.65it/s]


Epoch 5: Loss=0.2932 Accuracy=87.77%


## MobileNetV2 + Fashion-MNIST Results

### Classification Performance

| Epoch | Accuracy |
|--------|--------|
| 1 | 81.44% |
| 2 | 85.35% |
| 3 | 86.13% |
| 4 | 87.43% |
| 5 | 87.77% |

### Observation

MobileNetV2 achieved strong performance on Fashion-MNIST,
reaching 87.77% test accuracy.

This model-dataset combination was intentionally excluded
from meta-model training and will be used as the final
generalization benchmark.

In [52]:
mobilenet_feature_extractor_fashion = mobilenet_fashion.features

print("Feature Extractor Ready")

Feature Extractor Ready


In [53]:
import torch.nn.functional as F
import pandas as pd

mobilenet_fashion.eval()

confidences = []
entropies = []

activation_means = []
activation_stds = []
activation_maxs = []
activation_mins = []

failures = []

with torch.no_grad():

    for images, labels in fashion_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = mobilenet_fashion(images)

        probs = F.softmax(outputs, dim=1)

        confidence, predictions = torch.max(
            probs,
            dim=1
        )

        entropy = -torch.sum(
            probs * torch.log(probs + 1e-10),
            dim=1
        )

        activations = mobilenet_feature_extractor_fashion(
            images
        )

        activations = activations.view(
            activations.size(0),
            -1
        )

        activation_means.extend(
            activations.mean(dim=1).cpu().numpy()
        )

        activation_stds.extend(
            activations.std(dim=1).cpu().numpy()
        )

        activation_maxs.extend(
            activations.max(dim=1)[0].cpu().numpy()
        )

        activation_mins.extend(
            activations.min(dim=1)[0].cpu().numpy()
        )

        confidences.extend(
            confidence.cpu().numpy()
        )

        entropies.extend(
            entropy.cpu().numpy()
        )

        failures.extend(
            (predictions != labels)
            .int()
            .cpu()
            .numpy()
        )

mobilenet_fashion_df = pd.DataFrame({
    "confidence": confidences,
    "entropy": entropies,
    "activation_mean": activation_means,
    "activation_std": activation_stds,
    "activation_max": activation_maxs,
    "activation_min": activation_mins,
    "failure": failures
})

print("Shape:", mobilenet_fashion_df.shape)
print("Failures:", mobilenet_fashion_df["failure"].sum())

Shape: (10000, 7)
Failures: 1223


# Unseen Generalization Evaluation

## Objective

Evaluate the generalized meta-failure predictor on a
previously unseen dataset-architecture combination.

The meta-model was trained using:

- ResNet18 + CIFAR10
- MobileNetV2 + CIFAR10
- ResNet18 + FashionMNIST

The following combination was completely excluded from
training:

- MobileNetV2 + FashionMNIST

This experiment assesses the true generalization capability
of the proposed framework.

In [54]:
X_unseen = mobilenet_fashion_df.drop(
    columns=["failure"]
)

y_unseen = mobilenet_fashion_df["failure"]

print(X_unseen.shape)

(10000, 6)


In [55]:
unseen_preds = meta_model.predict(
    X_unseen
)

print("Predictions Generated")

Predictions Generated


In [56]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

print(
    "Unseen Meta Accuracy:",
    accuracy_score(
        y_unseen,
        unseen_preds
    )
)

print(
    classification_report(
        y_unseen,
        unseen_preds
    )
)

Unseen Meta Accuracy: 0.8774
              precision    recall  f1-score   support

           0       0.89      0.98      0.93      8777
           1       0.50      0.17      0.26      1223

    accuracy                           0.88     10000
   macro avg       0.70      0.58      0.60     10000
weighted avg       0.85      0.88      0.85     10000



# Final Unseen Generalization Results

## Evaluation Dataset

MobileNetV2 + FashionMNIST

This combination was completely excluded from meta-model
training and used only for final evaluation.

## Performance

| Metric | Value |
|----------|----------|
| Accuracy | 87.74% |
| Failure Precision | 50% |
| Failure Recall | 17% |
| Failure F1-Score | 26% |

## Observation

The generalized meta-model achieved strong overall
performance on an unseen dataset-architecture
combination.

This result demonstrates that the proposed framework
successfully transfers learned reliability patterns
across both datasets and architectures.

The framework maintained high overall accuracy despite
never observing this specific combination during training.

However, failure recall decreased compared to the
validation results, indicating that unseen failure
patterns remain challenging to identify.

## Conclusion

The experiment provides evidence that architecture-
independent reliability features can support cross-
dataset and cross-architecture failure prediction.

The framework demonstrates meaningful generalization
capability beyond the training combinations used during
meta-model development.

In [57]:
meta_df.to_csv(
    "../features/generalized_meta_dataset.csv",
    index=False
)

print("Meta Dataset Saved!")

Meta Dataset Saved!


In [58]:
import joblib

joblib.dump(
    meta_model,
    "../models/generalized_meta_model.pkl"
)

print("Meta Model Saved!")

Meta Model Saved!


In [59]:
feature_importance.to_csv(
    "../results/feature_importance.csv",
    index=False
)